# Регрессионный анализ: применимость линейной регрессии (Стадия 2)

**Проект:** Multi-Class Prediction of Cirrhosis Outcomes (Kaggle Playground S3E26).
**Предыдущий этап:** `Stage_1_analysis.ipynb` — EDA и очистка данных.

**Цель раздела:** проверить **возможность** применения линейной регрессии для предсказания клинического статуса пациента с циррозом печени (`Status`) по биохимическим анализам крови и анамнезу.

**Формат раздела — необходимый минимум:** проверка предпосылок линейной регрессии (блок 4) и вывод (блок 5). Обучение модели, метрики качества (R²/MAE/RMSE), интерпретация коэффициентов, стандартизация и корректная модель классификации (логистическая регрессия) вынесены в раздел 3: линейная регрессия — инструмент для непрерывных целей, и блок 4 показывает ещё до обучения, что наша цель такой не является.

**Постановка.** `Status` — категориальная переменная из трёх классов. Если бы линейная регрессия применялась, цель пришлось бы кодировать порядковой шкалой `C = 0` (жив/цензурирован), `CL = 1` (трансплантация), `D = 2` (смерть). Это сильное допущение — его корректность проверяется в блоке 4.

## **1. Импорты и настройки**

In [1]:
# Сторонние библиотеки
import os

import numpy as np
import pandas as pd

pd.set_option('display.width', 200)

## **2. Загрузка очищенных данных**

Данные берутся из `data/processed/train_clean.parquet` — это датасет `df_clean`, сохранённый в блоке 3.14 стадии Stage_1: без биологически невозможных значений, с инженерными признаками (`Age_years`, `Edema_num`, `is_severe`, `log_*`). Повторная очистка не требуется.

In [2]:
# Корень проекта: папка с data/, найденная вверх по дереву от текущей директории
def find_project_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(d, 'data')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError('Папка data/ не найдена вверх по дереву от ' + d)
        d = parent


PROJECT_ROOT = find_project_root()
df = pd.read_parquet(os.path.join(PROJECT_ROOT, 'data', 'processed', 'train_clean.parquet'))
print(f'Строк: {len(df)}, столбцов: {df.shape[1]}')
df.head(3)

Строк: 7905, столбцов: 30


,id,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,...,Age_years,Edema_num,is_severe,is_dead,log_Bilirubin,log_Copper,log_Alk_Phos,log_SGOT,log_Cholesterol,log_Tryglicerides
0,0,999,D-penicillamine,21532,M,N,N,N,N,2.3,...,59.0,0.0,1,1,1.193923,5.153292,7.379008,5.197392,5.758902,4.158883
1,1,2574,Placebo,19237,F,N,N,N,N,0.9,...,52.7,0.0,1,0,0.641854,4.158883,7.273093,4.911551,5.899898,4.488636
2,2,3428,Placebo,13727,F,N,Y,Y,Y,3.3,...,37.6,1.0,1,1,1.458615,4.882802,6.937314,4.790404,5.703783,3.931826


### Целевая переменная

In [3]:
# Кодируем Status в порядковую числовую шкалу: C=0, CL=1, D=2
status_map = {'C': 0, 'CL': 1, 'D': 2}
y = df['Status'].astype(str).map(status_map)

print('Распределение классов цели:')
display(y.value_counts(normalize=True).sort_index().rename('доля').round(4).to_frame())

Распределение классов цели:


,доля
Status,
0,0.6281
1,0.0348
2,0.3371


В матрицу признаков не входят: `id` (технический), `Status` и `is_dead` (целевая и её дубликат — утечка цели), `Drug` (неинформативен: φk со `Status` = 0.013, Cramér's V = 0.027 при χ² p = 0.053 — см. блок 3.13 стадии Stage_1), `Age` (дублирует `Age_years`), `Edema_num` и `is_severe` (дублируют one-hot `Edema` и `Stage`; проверка в блоке 2.5 Stage_1). Категориальные признаки кодируются one-hot с удалением базового уровня, чтобы избежать полной мультиколлинеарности dummy-переменных.

Исключение `Drug` — результат отбора неинформативных признаков, а не изменения данных: `data/processed/train_clean.parquet` остаётся полным артефактом и содержит все 30 столбцов.

In [4]:
drop_cols = ['id', 'Status', 'is_dead', 'Drug', 'Age', 'Edema_num', 'is_severe']
feature_cols = [c for c in df.columns if c not in drop_cols]

cat_cols = ['Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']
X = pd.get_dummies(df[feature_cols], columns=cat_cols, drop_first=True).astype(float)

print(f'Матрица признаков: {X.shape[0]} строк x {X.shape[1]} столбцов')
print('Столбцы:', list(X.columns))

Матрица признаков: 7905 строк x 24 столбцов
Столбцы: ['N_Days', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin', 'Stage', 'Age_years', 'log_Bilirubin', 'log_Copper', 'log_Alk_Phos', 'log_SGOT', 'log_Cholesterol', 'log_Tryglicerides', 'Sex_M', 'Ascites_Y', 'Hepatomegaly_Y', 'Spiders_Y', 'Edema_S', 'Edema_Y']


## **4. Быстрая оценка применимости линейной регрессии**

Линейная регрессия имеет набор предпосылок, которые можно проверить за секунды — **без** обучения модели, train/test-разбиения и анализа остатков. Если предпосылки грубо нарушены, результат моделирования предсказуем заранее. Проверяем четыре вещи на всей очищенной выборке (модели нет, поэтому утечка данных не грозит):

1. **Тип и распределение цели** — линейная регрессия рассчитана на непрерывную цель на всей числовой прямой.
2. **Наличие линейной связи** «признак ↔ цель» (корреляция Пирсона).
3. **Линейность этой связи** — сравнение Пирсона и Спирмена.
4. **Мультиколлинеарность** между независимыми признаками (пары «признак ↔ его log-копия» исключаются: они коллинеарны по построению и являются артефактом инжиниринга, а не свойством данных).

> **Пирсон vs Спирмен.** Пирсон меряет силу *линейной* связи, Спирмен — силу *монотонной* (любой возрастающей/убывающей). Если Спирмен заметно больше Пирсона, связь есть, но она нелинейна — линейная регрессия её недоиспользует.

In [5]:
# Проверка 1. Тип и распределение цели
n_classes = y.nunique()
print(f'Целевая переменная принимает {n_classes} значения: {dict(y.value_counts().sort_index())}')

if n_classes <= 10:
    print('=> Цель ДИСКРЕТНАЯ (мало значений). Линейная регрессия формально')
    print('   не предназначена для такой цели: она вернёт значения вне {0, 1, 2},')
    print('   а остатки не будут нормальными. Это ограничение, а не ошибка данных.')
else:
    print('=> Цель почти непрерывная — предпосылка линейной регрессии выполнена.')

Целевая переменная принимает 3 значения: {0: np.int64(4965), 1: np.int64(275), 2: np.int64(2665)}
=> Цель ДИСКРЕТНАЯ (мало значений). Линейная регрессия формально
   не предназначена для такой цели: она вернёт значения вне {0, 1, 2},
   а остатки не будут нормальными. Это ограничение, а не ошибка данных.


In [6]:
# Проверка 2. Линейная связь каждого признака с целью (Пирсон)
corr_with_target = X.corrwith(y).sort_values(key=abs, ascending=False)
print('Топ-8 линейных связей с целью:')
display(corr_with_target.head(8).round(3).to_frame('r (Пирсон)'))

max_abs = corr_with_target.abs().max()
if max_abs < 0.2:
    verdict2 = 'слабые: почти все |r| < 0.2 — линейной модели брать практически нечего'
elif max_abs < 0.4:
    verdict2 = 'умеренные: |r| ~ 0.2-0.4 — линейная связь есть, но сигнал не сильный'
else:
    verdict2 = 'сильные: есть признаки с |r| > 0.4 — линейная модель вытянет заметную часть'
print(f'\n=> Связи с целью: {verdict2}. Максимум |r| = {max_abs:.2f} ({corr_with_target.abs().idxmax()}).')

Топ-8 линейных связей с целью:


,r (Пирсон)
log_Bilirubin,0.534
Bilirubin,0.429
log_Copper,0.414
Hepatomegaly_Y,0.396
Prothrombin,0.388
Copper,0.386
Stage,0.347
N_Days,-0.343



=> Связи с целью: сильные: есть признаки с |r| > 0.4 — линейная модель вытянет заметную часть. Максимум |r| = 0.53 (log_Bilirubin).


In [7]:
# Проверка 3. Линейность связи: Пирсон vs Спирмен для топ-признаков
top_feats = corr_with_target.head(6).index
lin_check = pd.DataFrame({
    'Пирсон': [X[f].corr(y) for f in top_feats],
    'Спирмен': [X[f].corr(y, method='spearman') for f in top_feats],
}, index=top_feats)
lin_check['разрыв'] = (lin_check['Спирмен'] - lin_check['Пирсон']).abs()
display(lin_check.round(3))

print('=> Если разрыв большой (напр. > 0.05), связь нелинейна.')
print('   Исходный Bilirubin: разрыв заметен (нелинеен), а log_Bilirubin почти совпадает')
print('   с Спирменом -> лог-преобразование делает связь линейной и пригодной для модели.')

,Пирсон,Спирмен,разрыв
log_Bilirubin,0.534,0.540,0.006
Bilirubin,0.429,0.540,0.111
log_Copper,0.414,0.438,0.024
Hepatomegaly_Y,0.396,0.397,0.000
Prothrombin,0.388,0.382,0.006
Copper,0.386,0.438,0.052


=> Если разрыв большой (напр. > 0.05), связь нелинейна.
   Исходный Bilirubin: разрыв заметен (нелинеен), а log_Bilirubin почти совпадает
   с Спирменом -> лог-преобразование делает связь линейной и пригодной для модели.


In [8]:
# Проверка 4. Мультиколлинеарность между независимыми признаками
# (исключаем тривиальные пары «исходный признак ↔ его log-копия» —
#  они коллинеарны по построению и не отражают реальную структуру данных)
def is_trivial_duplicate(pair):
    a, b = pair
    return a == f'log_{b}' or b == f'log_{a}'

feat_corr = X.corr()
mask = np.triu(np.ones_like(feat_corr, dtype=bool), k=1)
pairs = feat_corr.where(mask).stack().sort_values(key=abs, ascending=False)

# Исключаем тривиальные пары (артефакт инжиниринга, а не данные)
unique_pairs = pairs[[not is_trivial_duplicate(p) for p in pairs.index]]
print('Топ-6 пар независимых признаков по |r| (без тривиальных log-копий):')
display(unique_pairs.head(6).round(3).to_frame('|r|'))

n_strong = int((unique_pairs > 0.8).sum())
if n_strong:
    print(f'\n=> Найдено {n_strong} пар с |r| > 0.8 — реальная мультиколлинеарность.')
else:
    print(f'\n=> Реальной мультиколлинеарности (|r| > 0.8) НЕТ.')
    print(f'Сильнейшая пара: {unique_pairs.index[0]} |r| = {unique_pairs.iloc[0]:.3f}')
    print('Это допустимо — проблема начинается от ~0.85-0.9.')

# Для полноты показываем, сколько тривиальных пар было бы без фильтра
trivial_count = int((pairs > 0.8).sum()) - n_strong
print(f'\n(Всего пар с |r| > 0.8, включая тривиальные: {trivial_count} — это лог-копии)')

Топ-6 пар независимых признаков по |r| (без тривиальных log-копий):


,,|r|
Ascites_Y,Edema_Y,0.658
log_Bilirubin,log_Copper,0.538
Copper,log_Bilirubin,0.510
Stage,Hepatomegaly_Y,0.498
log_Bilirubin,log_SGOT,0.464
SGOT,log_Bilirubin,0.451



=> Реальной мультиколлинеарности (|r| > 0.8) НЕТ.
Сильнейшая пара: ('Ascites_Y', 'Edema_Y') |r| = 0.658
Это допустимо — проблема начинается от ~0.85-0.9.

(Всего пар с |r| > 0.8, включая тривиальные: 6 — это лог-копии)


### Вердикт по предпосылкам (до всякого обучения модели)


По четырём быстрым проверкам заранее видно:

| Предпосылка | Что показали метрики | Вывод для линейной регрессии |
|---|---|---|
| Непрерывность цели | 3 значения (0/1/2) | **Грубо нарушена**: предсказания выйдут за диапазон классов, остатки не будут нормальными |
| Линейная связь с целью | максимум \|r\| ≈ 0.53 (`log_Bilirubin`) | Сигнал есть — признаки информативны, но связность умеренная |
| Линейность связи | у `Bilirubin` разрыв Пирсон/Спирмен ≈ 0.11, у `log_Bilirubin` ≈ 0.01 | Исходные лабораторные признаки нелинейны относительно цели; лог-копии — да |
| Мультиколлинеарность | между независимыми признаками \|r\| ≤ 0.66 (сильнейшая пара `Ascites_Y ↔ Edema_Y`) | Предпосылка выполнена — структура данных коллинеарных пар не содержит |

Главный пункт — первый, и он неустраним: **дискретная категориальная цель не является непрерывной величиной**. Никакая работа с признаками (логи, отбор, масштабирование) это не лечит, потому что нарушение — в самой постановке задачи, а не в данных.

> **Вывод по методике:** линейную регрессию для задач классификации не используют. Корректный инструмент для предсказания `Status` — многоклассовая **логистическая регрессия** (и далее ансамбли), с метриками классификации. Быстрая проверка заняла ~30 секунд и дала тот же управленческий вывод, что занял бы полный цикл моделирования.

## **5. Выводы**

### Ответ на вопрос задачи

**Можно ли применить линейную регрессию для предсказания клинического статуса?** Проверка предпосылок (блок 4) даёт отрицательный ответ, и он следует из структуры задачи, а не из качества данных:

1. **Цель не является непрерывной.** `Status` принимает три значения, разделённые произвольным числовым шагом. Линейная регрессия по природе выдаёт любое вещественное число: предсказания вроде −0.4 или 2.7 не соответствуют ни одному классу, а промежуточные значения (1.3) не имеют клинического смысла — «полутрансплантации» не существует.

2. **Порядковая кодировка вводит ложную метрику.** Модель была бы вынуждена считать, что расстояние C→CL равно CL→D. Трансплантация (`CL`, всего 3.5 % наблюдений) — не «промежуточная тяжесть» между выживанием и смертью, а отдельный исход, зависящий в том числе от решения врача и доступности донорского органа.

3. **Метрики качества регрессии были бы неинтерпретируемы.** R², MAE и RMSE измеряют ошибку в единицах условной шкалы 0–2; «ошибка 0.5» не отвечает на практический вопрос «выживёт ли пациент». Для классификации нужны accuracy / log-loss / ROC AUC.

4. **При этом сами данные для моделирования пригодны.** Признаки информативны (максимум \|r\| = 0.53 у `log_Bilirubin`), линейная связность подтверждена сравнением Пирсона/Спирмена, реальной мультиколлинеарности между независимыми признаками нет (сильнейшая пара \|r\| = 0.66 — `Ascites_Y ↔ Edema_Y`, клинически ожидаемое сочетание маркёров декомпенсации). Вывод «нет» относится к выбору метода, а не к датасету.

5. **Практический итог метода:** отказ от линейной регрессии здесь принят за ~30 секунд корреляционных расчётов, без единого вызова `.fit()`. Это и есть ценность проверки предпосылок: она экономит цикл моделирования и защищает от некорректной интерпретации его результатов.


### Глоссарий терминов

| Термин | Что означает | Зачем нужен |
|--------|-------------|-------------|
| **Целевая переменная** | То, что модель пытается предсказать (здесь — `Status`) | Без неё модель не знает, чему учиться |
| **Признаки (features)** | Входы модели: лабораторные показатели, стадия, демография (24 столбца) | Информация, на основе которой строится предсказание |
| **Категориальная (номинальная) цель** | Цель задаётся конечным набором классов без метрического расстояния между ними | Определяет выбор класса моделей: нужны классификаторы, а не регрессии |
| **Порядковая (ordinal) кодировка** | Преобразование `C→0, CL→1, D→2`: числа упорядочены, но «шаг» между ними произволен | Единственный способ подать категорию в линейную регрессию — и источник её некорректности здесь |
| **Предпосылка модели** | Свойство данных, при котором метод корректен (непрерывность цели, линейность связи, отсутствие мультиколлинеарности) | Проверка предпосылок отвечает на вопрос «можно ли применять метод» до вопроса «насколько качественно» |
| **Корреляция Пирсона** | Сила *линейной* связи, \|r\| ≤ 1 | Показывает, есть ли что «вытягивать» линейной модели |
| **Корреляция Спирмена** | Сила *монотонной* связи (ранговая) | Сравнение с Пирсоном выявляет нелинейность: большой разрыв = связь есть, но не линейная |
| **Мультиколлинеарность** | Высокая корреляция между признаками | Делает коэффициенты нестабильными: знак может «переворачиваться» |
| **Тривиальная коллинеарность** | Копия признака, созданная самим аналитиком (`Bilirubin` ↔ `log_Bilirubin`) | Артефакт инжиниринга, а не свойство данных — её нужно фильтровать, иначе она маскирует реальные связи |
| **One-hot кодирование** | categorical-признак заменяется бинарными столбцами по уровням (`Sex_M`) | Делает категории пригодными для числовых моделей |
| **Утечка цели (target leakage)** | В признаки попадает информация, прямо вычисляемая из цели (`is_dead` — это `Status == 'D'`) | Модель «предсказывает» известное, метрики становятся фиктивными |